# 03 — Business Analysis

Answers the business questions in `business_questions.docx`. All visuals saved to `visuals/` for use in `final_insights.docx`.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATA_PROCESSED
from src.viz import PALETTE, save, barh_ranked, line_trend

sns.set_style("whitegrid")

In [ ]:
orders_f = pd.read_parquet(DATA_PROCESSED / "features_orders.parquet")
customers_f = pd.read_parquet(DATA_PROCESSED / "features_customers.parquet")
products_f = pd.read_parquet(DATA_PROCESSED / "features_products.parquet")
sellers_f = pd.read_parquet(DATA_PROCESSED / "features_sellers.parquet")
reviews = pd.read_parquet(DATA_PROCESSED / "reviews.parquet")
payments = pd.read_parquet(DATA_PROCESSED / "payments.parquet")
customers_raw = pd.read_parquet(DATA_PROCESSED / "customers.parquet")

delivered = orders_f[orders_f['order_status'] == 'delivered'].copy()

In [ ]:
kpis = {
    'Delivered orders': f"{delivered['order_id'].nunique():,}",
    'Total revenue': f"R$ {delivered['order_value'].sum():,.0f}",
    'Avg order value': f"R$ {delivered['order_value'].mean():,.2f}",
    'Unique customers': f"{customers_f.shape[0]:,}",
    'Active sellers': f"{sellers_f.shape[0]:,}",
    'Late delivery rate': f"{delivered['is_late'].mean():.1%}",
}
for k, v in kpis.items():
    print(f"{k:22s} {v}")

In [ ]:
monthly = (
    delivered.groupby('order_month')
    .agg(revenue=('order_value', 'sum'),
         orders=('order_id', 'nunique'))
    .reset_index()
)
monthly = monthly[
    (monthly['order_month'] >= '2017-01') &
    (monthly['order_month'] < '2018-09')
].copy()
monthly['dt'] = pd.to_datetime(monthly['order_month'] + '-01')

fig, ax = line_trend(
    monthly, 'dt', 'revenue',
    'Monthly Revenue Trend (2017–2018)',
    'Month', 'Revenue (BRL)',
    annotate=[(pd.Timestamp('2017-11-01'), 'Black Friday 2017')],
)
save(fig, '03_monthly_revenue')
plt.show()

monthly['mom_growth_pct'] = monthly['revenue'].pct_change() * 100
monthly[['order_month', 'revenue', 'mom_growth_pct']].round(2).tail(8)

In [ ]:
cat_rev = (
    products_f.groupby('category')
    .agg(revenue=('revenue', 'sum'), orders=('orders', 'sum'))
    .reset_index()
    .sort_values('revenue', ascending=False)
    .head(12)
)

fig, ax = barh_ranked(
    cat_rev, 'category', 'revenue',
    'Top 12 Categories by Revenue',
    'Revenue (BRL)',
)
save(fig, '03_top_categories')
plt.show()

In [ ]:
geo = (
    delivered.merge(
        customers_raw[['customer_id', 'customer_state']],
        on='customer_id', how='left'
    )
    .groupby('customer_state')
    .agg(revenue=('order_value', 'sum'),
         orders=('order_id', 'nunique'))
    .reset_index()
    .sort_values('revenue', ascending=False)
    .head(10)
)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(geo['customer_state'], geo['revenue'],
        color=PALETTE['primary'], alpha=0.85)
ax1.set_ylabel('Revenue (BRL)', color=PALETTE['primary'])
ax1.set_title('Top 10 States: Revenue vs Order Volume')
ax2 = ax1.twinx()
ax2.plot(geo['customer_state'], geo['orders'],
         color=PALETTE['accent'], marker='o', linewidth=2)
ax2.set_ylabel('Orders', color=PALETTE['accent'])
save(fig, '03_geo_distribution')
plt.show()

In [ ]:
buckets = (
    delivered.assign(
        bucket=pd.cut(
            delivered['delivery_delay_days'],
            bins=[-np.inf, -1, 0, 3, 7, np.inf],
            labels=['Early', 'On Time', '1-3d Late', '4-7d Late', '7d+ Late'],
        )
    )
    .groupby('bucket', observed=True)
    .agg(orders=('order_id', 'count'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=buckets, x='bucket', y='orders',
            palette='RdYlGn_r', ax=ax, hue='bucket', legend=False)
ax.set_title('Orders by Delivery Performance Bucket')
ax.set_ylabel('Orders')
save(fig, '03_delivery_buckets')
plt.show()

merged = delivered.merge(
    reviews[['order_id', 'review_score']], on='order_id', how='inner'
)
merged['bucket'] = pd.cut(
    merged['delivery_delay_days'],
    bins=[-np.inf, -1, 0, 3, 7, np.inf],
    labels=['Early', 'On Time', '1-3d Late', '4-7d Late', '7d+ Late'],
)

avg_by_bucket = (
    merged.groupby('bucket', observed=True)['review_score']
    .agg(['mean', 'count'])
    .round(2)
    .reset_index()
)
display(avg_by_bucket)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=avg_by_bucket, x='bucket', y='mean',
            palette='RdYlGn_r', ax=ax, hue='bucket', legend=False)
ax.set_title('Average Review Score by Delivery Performance')
ax.set_ylabel('Avg Review Score')
ax.set_ylim(1, 5)
save(fig, '03_delay_vs_reviews')
plt.show()

In [ ]:
pay = (
    payments.groupby('payment_type')
    .agg(
        payments=('order_id', 'count'),
        avg_installments=('payment_installments', 'mean'),
        total_value=('payment_value', 'sum'),
    )
    .reset_index()
    .sort_values('total_value', ascending=False)
)
display(pay.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=pay, x='payment_type', y='total_value',
            palette='viridis', ax=ax, hue='payment_type', legend=False)
ax.set_title('Total Value by Payment Method')
ax.set_ylabel('BRL')
save(fig, '03_payment_types')
plt.show()

In [ ]:
dist = reviews['review_score'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#27AE60']
bars = ax.bar(dist.index.astype(str), dist.values, color=colors)
ax.set_title('Review Score Distribution')
ax.set_xlabel('Review Score'); ax.set_ylabel('Reviews')
for bar, val in zip(bars, dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 200,
            f'{val:,}', ha='center', fontsize=9)
save(fig, '03_review_distribution')
plt.show()

print(f"Average review score: {reviews['review_score'].mean():.2f}")

## Key Insights

1. **Revenue is concentrated** in SP, RJ, MG; SP dominates both revenue and orders
2. **Black Friday 2017** is the single biggest month — plan inventory and logistics around it
3. **Late deliveries** drop average review scores from ~4.3 to ~2.5 (a 40%+ reduction)
4. **Credit card** is the dominant payment method, with **boleto** as a distant second
5. **Reviews are bimodal** — mostly 5-star, but 1-star is the second most common
6. **Top categories**: bed_bath_table, health_beauty, sports_leisure